In [15]:
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

nlp = spacy.load("en_core_web_sm")

# Step 1: Load the complaint dataset
complaint_data = pd.read_csv("complaintdata.csv")  # Replace with your actual dataset path


In [16]:
# Step 2: Preprocess the feedback data
def preprocess_feedback(feedback_text):
    # Lowercasing and basic preprocessing
    feedback_text = feedback_text.lower()
    return feedback_text

complaint_data['processed_feedback'] = complaint_data['feedback'].apply(preprocess_feedback)

In [17]:
# Step 3: Named Entity Recognition (NER) to find time-related entities and issues
def extract_time_and_action_entities(text):
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if ent.label_ == "TIME":  # Extract time-related entities (e.g., "2 hours", "yesterday")
            entities.append(ent.text)
        if ent.label_ == "EVENT":  # Extract events or actions that indicate areas needing improvement
            entities.append(ent.text)
    return entities

complaint_data['time_and_actions'] = complaint_data['processed_feedback'].apply(extract_time_and_action_entities)

In [18]:
# Step 4: Keyword Extraction using TF-IDF for central idea
def extract_central_idea(text):
    # Ensure that the text is not empty or consists of only stop words
    if not text.strip():
        return "No central idea"

    # Vectorizing the text
    vectorizer = TfidfVectorizer(max_features=10, stop_words='english')  # Limit to top 10 features
    try:
        tfidf_matrix = vectorizer.fit_transform([text])

        # Check if the vocabulary is empty (only stopwords or empty text)
        if tfidf_matrix.shape[1] == 0:
            return "No central idea"

        feature_names = vectorizer.get_feature_names_out()
        feature_scores = tfidf_matrix.sum(axis=0).A1  # Summing TF-IDF scores for each word
        sorted_indices = feature_scores.argsort()[-3:][::-1]  # Get top 3 words by score

        top_keywords = [feature_names[i] for i in sorted_indices]
        return " ".join(top_keywords)
    except ValueError as e:
        return "No central idea"

complaint_data['central_idea'] = complaint_data['processed_feedback'].apply(extract_central_idea)

In [19]:
# Step 5: Apply Sentiment Analysis (Using the pre-trained model)
# Load the sentiment analysis model
sentiment_model = joblib.load("sentiment_model.pkl")
vectorizer = joblib.load("tfidf_vectorizer.pkl")

def predict_sentiment(feedback):
    processed_feedback = preprocess_feedback(feedback)
    vectorized_feedback = vectorizer.transform([processed_feedback])
    sentiment_probabilities = sentiment_model.predict_proba(vectorized_feedback)[0]
    sentiment_score = sentiment_probabilities[1]  # Positive sentiment probability
    return sentiment_score

complaint_data['sentiment'] = complaint_data['feedback'].apply(predict_sentiment)

In [20]:

# Step 6: Detect areas needing improvement based on sentiment and extracted central idea
def detect_improvement_area(row):
    # Threshold for negative sentiment (e.g., 0.4 means sentiment score below 0.4 indicates negative feedback)
    sentiment_threshold = 0.4
    if row['sentiment'] < sentiment_threshold:
        return f"Improvement needed due to negative sentiment and issue detected: {row['central_idea']}"
    return "No improvement needed"

complaint_data['improvement_area'] = complaint_data.apply(detect_improvement_area, axis=1)

# Step 7: Extract and Save the areas needing improvement
areas_needing_improvement = complaint_data[complaint_data['improvement_area'].str.contains("Improvement needed")]

In [21]:
# Save the areas needing improvement
areas_needing_improvement.to_csv("areas_needing_improvement.csv", index=False)
print("Areas needing improvement detected and saved to 'areas_needing_improvement.csv'")

Areas needing improvement detected and saved to 'areas_needing_improvement.csv'
